# AKOrN on CLEVRTex — Multi-Session Training Notebook

**Target:** Table 1 row "AKOrN (attn, L=1)" on CLEVRTex — FG-ARI 75.6, MBO 55.0  
**Platform:** Kaggle T4×2, 12h session cap  
**Repo:** github.com/DimitriBnvl/AKOrN (fork of autonomousvision/akorn)

---

## Multi-Session Workflow

Training 500 epochs (~32–40 hours total) requires multiple Kaggle sessions:

1. **Session 1 (fresh start):** Leave `PREV_CKPT_INPUT` pointing to a non-existent path (or an empty dataset).  
   Click **Save & Run All** (Commit). Kaggle runs the notebook and saves everything under `/kaggle/working/` as version output.

2. **Session 2+ (resume):**  
   - Go to your notebook → **Data** → **Add input** → select the *output dataset* of the previous version.  
   - Update `PREV_CKPT_INPUT` to match that dataset's mount path (e.g. `/kaggle/input/akorn-clvtex-ckpts`).  
   - Cell 2 will copy the latest checkpoints into `RUN_DIR` before training starts.  
   - Cell 3 auto-detects the latest checkpoint and sets `FINETUNE_ARG` / `START_EPOCH`.  
   - Cell 5 resumes training from `START_EPOCH` automatically.

3. **Repeat** until epoch 499 is reached, then run Cell 7 for evaluation.

### Dataset prerequisites
Upload the 5 CLEVRTex tar.gz parts (`clevrtex_full_part1.tar.gz` … `part5.tar.gz`) to a Kaggle Dataset and set `DATA_ROOT` to its mount path. The `CLEVRTEX` dataset class will look for a `clevrtex_full/` subfolder under that path automatically.

## Cell 1 — Configuration
All paths and hyperparameters in one place. Edit these before committing.

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
REPO_DIR        = "/kaggle/working/AKOrN"
DATA_ROOT       = "/kaggle/input/clevrtex-full"       # set to your Kaggle Dataset name
PREV_CKPT_INPUT = "/kaggle/input/akorn-clvtex-ckpts"  # set to previous version output, or leave absent to start fresh

# ── Experiment ─────────────────────────────────────────────────────────────────
RUN_NAME = "clvtex_akorn"
RUN_DIR  = f"{REPO_DIR}/runs/{RUN_NAME}"

# ── Hyperparameters (match README / Table 1 row) ───────────────────────────────
EPOCHS      = 500
BATCHSIZE   = 256
LR          = 1e-3
CKPT_EVERY  = 50
NUM_GPUS    = 2

print("Configuration loaded.")
print(f"  REPO_DIR        = {REPO_DIR}")
print(f"  DATA_ROOT       = {DATA_ROOT}")
print(f"  PREV_CKPT_INPUT = {PREV_CKPT_INPUT}")
print(f"  RUN_DIR         = {RUN_DIR}")
print(f"  EPOCHS={EPOCHS}  BATCHSIZE={BATCHSIZE}  LR={LR}  CKPT_EVERY={CKPT_EVERY}  NUM_GPUS={NUM_GPUS}")

## Cell 2 — Setup (idempotent)
Clone repo, install deps, create run dir, and restore checkpoints from a previous session.

In [ ]:
import os
import glob
import shutil
import subprocess

# ── 1. Clone repository (skip if already present) ─────────────────────────────
if not os.path.isdir(REPO_DIR):
    print("Cloning AKOrN repository...")
    subprocess.run(
        ["git", "clone", "https://github.com/DimitriBnvl/AKOrN.git", REPO_DIR],
        check=True,
    )
    print("Clone complete.")
else:
    print(f"Repository already present at {REPO_DIR}, skipping clone.")

# ── 2. Install Python dependencies ────────────────────────────────────────────
print("\nInstalling requirements...")
subprocess.run(
    ["pip", "install", "-q", "-r", os.path.join(REPO_DIR, "requirements.txt")],
    check=True,
)
print("Requirements installed.")

# ── 3. Create run directory ───────────────────────────────────────────────────
os.makedirs(RUN_DIR, exist_ok=True)
print(f"\nRun directory ready: {RUN_DIR}")

# ── 4. Restore checkpoints from previous session ──────────────────────────────
if os.path.isdir(PREV_CKPT_INPUT):
    prev_ckpts = glob.glob(os.path.join(PREV_CKPT_INPUT, "**", "*.pth"), recursive=True)
    if prev_ckpts:
        print(f"\nFound {len(prev_ckpts)} checkpoint(s) in {PREV_CKPT_INPUT}:")
        for src in sorted(prev_ckpts):
            dst = os.path.join(RUN_DIR, os.path.basename(src))
            if os.path.exists(dst):
                print(f"  [skip] {os.path.basename(src)} already in RUN_DIR")
            else:
                shutil.copy2(src, dst)
                print(f"  [copy] {os.path.basename(src)}  ({os.path.getsize(src) / 1e6:.1f} MB)")
    else:
        print(f"\nNo .pth files found under {PREV_CKPT_INPUT} — starting fresh.")
else:
    print(f"\nPREV_CKPT_INPUT not found ({PREV_CKPT_INPUT}) — starting fresh.")

print("\nSetup complete.")

## Cell 3 — Verification Gate
Checks GPUs, data, and checkpoint state before committing any compute.

In [ ]:
import glob
import os
import torch

errors = []

# ── Check 1: exactly 2 GPUs ───────────────────────────────────────────────────
n_gpu = torch.cuda.device_count()
if n_gpu != 2:
    errors.append(f"Expected 2 GPUs, found {n_gpu}. Select 'GPU T4 x2' accelerator in notebook settings.")

# ── Check 2: both GPUs are T4 (sm_75+, required for Flash SDP) ────────────────
if n_gpu >= 1:
    for i in range(n_gpu):
        name = torch.cuda.get_device_name(i)
        if "T4" not in name:
            errors.append(
                f"GPU {i} is '{name}' — P100/Pascal lacks Flash SDP (sm_75+), use T4 x2."
            )

# ── Check 3: CLEVRTex data is present ────────────────────────────────────────
expected_data_dir = os.path.join(DATA_ROOT, "clevrtex_full")
if not os.path.isdir(expected_data_dir):
    errors.append(
        f"CLEVRTex data not found at '{expected_data_dir}'. "
        f"Ensure the Kaggle Dataset is attached and DATA_ROOT='{DATA_ROOT}' is correct."
    )

# ── Check 4 & 5: find latest checkpoint and set resume variables ───────────────
ckpt_pattern = os.path.join(RUN_DIR, "checkpoint_*.pth")
existing_ckpts = sorted(glob.glob(ckpt_pattern))

if existing_ckpts:
    FINETUNE_ARG = existing_ckpts[-1]                            # absolute path
    _meta        = torch.load(FINETUNE_ARG, weights_only=True)
    START_EPOCH  = int(_meta["epoch"]) + 1                      # resume from next epoch
    resume_msg   = f"Resume from epoch {START_EPOCH} (checkpoint: {os.path.basename(FINETUNE_ARG)})"
else:
    FINETUNE_ARG = None
    START_EPOCH  = 0
    resume_msg   = "No checkpoint found — training from scratch (epoch 0)"

# ── Print all errors then raise if any ───────────────────────────────────────
if errors:
    print("VERIFICATION FAILED:")
    for e in errors:
        print(f"  [ERROR] {e}")
    raise RuntimeError(f"{len(errors)} verification error(s). See messages above.")

# ── Summary ──────────────────────────────────────────────────────────────────
print("Verification passed.")
print()
for i in range(n_gpu):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  "
          f"{props.total_memory / 1e9:.1f} GB  sm_{props.major}{props.minor}")
print(f"  Data  : {expected_data_dir}")
print(f"  {resume_msg}")
print(f"  FINETUNE_ARG = {FINETUNE_ARG}")
print(f"  START_EPOCH  = {START_EPOCH}")
print(f"  Epochs remaining: {EPOCHS - START_EPOCH} / {EPOCHS}")

## Cell 4 — Timing Probe
Single-GPU forward+backward pass to estimate wall-clock time per epoch and total sessions needed.

In [ ]:
import sys
import os
import time
import torch
import torch.nn as nn

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

from source.models.objs.knet import AKOrN

# ── Build model (matches Table 1 / README config) ─────────────────────────────
net = AKOrN(
    4, ch=256, L=1, T=8, gamma=1.0, J="attn",
    use_omega=False, global_omg=False, c_norm="gn",
    psize=8, imsize=128, autorescale=False,
    init_omg=0.01, learn_omg=False,
    maxpool=True, project=True, heads=8,
    use_ro_x=False, no_ro=False, gta=True,
).cuda()

opt = torch.optim.Adam(net.parameters(), lr=LR)

# Batch: train_obj.py uses batchsize // num_processes per GPU, then augmentation
# doubles images (2 views per image), so the tensor seen per GPU is:
#   (BATCHSIZE // NUM_GPUS) images × 2 views = (BATCHSIZE // NUM_GPUS * 2) frames
per_gpu_images = BATCHSIZE // NUM_GPUS          # 128
fake = torch.randn(per_gpu_images * 2, 3, 128, 128, device="cuda")  # 2-view batch

WARMUP_STEPS = 3
TIMED_STEPS  = 20

torch.backends.cudnn.benchmark = True
torch.backends.cuda.enable_flash_sdp(enabled=True)

net.train()

# Warm-up
for _ in range(WARMUP_STEPS):
    out = net(fake)
    # Proxy loss: use out.mean() instead of SimCLR to avoid cross-process gather
    # (SimCLR calls all_gather which requires a real DDP process group).
    # This measures forward+backward compute time only; loss shape doesn't affect timing.
    loss = out.mean()
    opt.zero_grad()
    loss.backward()
    opt.step()

torch.cuda.synchronize()
t0 = time.perf_counter()

for _ in range(TIMED_STEPS):
    out = net(fake)
    loss = out.mean()  # proxy loss — see comment above
    opt.zero_grad()
    loss.backward()
    opt.step()

torch.cuda.synchronize()
t1 = time.perf_counter()

# ── Compute estimates ─────────────────────────────────────────────────────────
secs_per_step  = (t1 - t0) / TIMED_STEPS

# Training images: ~40,000 in CLEVRTex train split
TRAIN_IMAGES    = 40_000
STEPS_PER_EPOCH = (TRAIN_IMAGES // NUM_GPUS) // (BATCHSIZE // NUM_GPUS)  # = 156

# DDP adds communication overhead vs single-GPU timing (~20%)
DDP_OVERHEAD    = 1.2

secs_per_epoch  = secs_per_step * DDP_OVERHEAD * STEPS_PER_EPOCH
mins_per_epoch  = secs_per_epoch / 60.0
total_hours     = secs_per_epoch * EPOCHS / 3600.0
remaining_hours = secs_per_epoch * (EPOCHS - START_EPOCH) / 3600.0

SESSION_CAP_H   = 11.5  # Kaggle hard-kills at 12h; leave 30 min margin
sessions_needed = remaining_hours / SESSION_CAP_H

print("Timing probe results")
print(f"  Steps per epoch  (formula): {STEPS_PER_EPOCH}")
print(f"  s/step  (single GPU, measured): {secs_per_step:.3f}")
print(f"  s/step  (2-GPU est. w/ {DDP_OVERHEAD:.1f}x DDP overhead): {secs_per_step * DDP_OVERHEAD:.3f}")
print(f"  min/epoch (est.):  {mins_per_epoch:.2f}")
print(f"  Total hours for {EPOCHS} ep (est.):     {total_hours:.1f} h")
print(f"  Remaining hours from ep {START_EPOCH} (est.): {remaining_hours:.1f} h")
print(f"  Sessions needed at {SESSION_CAP_H}h cap: {sessions_needed:.1f}")

MAX_SESSIONS = 8
assert sessions_needed < MAX_SESSIONS, (
    f"Estimated {sessions_needed:.1f} sessions needed exceeds {MAX_SESSIONS}-session sanity limit. "
    f"Consider reducing EPOCHS or increasing BATCHSIZE. "
    f"Current: EPOCHS={EPOCHS}, START_EPOCH={START_EPOCH}, remaining={remaining_hours:.1f}h."
)

# ── Clean up before training ──────────────────────────────────────────────────
del net, opt, fake
torch.cuda.empty_cache()
print("\nGPU memory released. Ready for training.")

## Cell 5 — Training
Launch DDP training via `accelerate`. Output is streamed line by line.

In [ ]:
import sys
import os
import subprocess

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)  # train_obj.py writes 'runs/<exp_name>/' relative to cwd

# ── Build accelerate launch command ───────────────────────────────────────────
cmd = [
    "accelerate", "launch",
    "--multi_gpu",
    f"--num_processes={NUM_GPUS}",
    "train_obj.py",
    f"--exp_name={RUN_NAME}",
    f"--data_root={DATA_ROOT}",   # CLEVRTEX class appends clevrtex_full/ subfolder automatically
    "--model=akorn",
    "--data=clevrtex_full",
    "--J=attn",
    "--L=1",
    f"--epochs={EPOCHS}",
    f"--batchsize={BATCHSIZE}",
    f"--lr={LR}",
    f"--checkpoint_every={CKPT_EVERY}",
]

# Append resume checkpoint if one was detected in Cell 3
if FINETUNE_ARG is not None:
    cmd.append(f"--finetune={FINETUNE_ARG}")

print("Training command:")
print(" ".join(cmd))
print(f"\nResuming from epoch: {START_EPOCH} / {EPOCHS}")
print("-" * 72)

# ── Stream subprocess output line by line ─────────────────────────────────────
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=REPO_DIR,
)

for line in proc.stdout:
    print(line, end="", flush=True)

proc.wait()

if proc.returncode != 0:
    raise subprocess.CalledProcessError(proc.returncode, cmd)

print("-" * 72)
print("Training completed successfully.")

## Cell 6 — Checkpoint Status
Inventory of all saved checkpoints and EMA files. Informational only — never raises.

In [ ]:
import glob
import os
import torch

print(f"Checkpoint inventory: {RUN_DIR}")
print("-" * 72)

ckpt_files = sorted(glob.glob(os.path.join(RUN_DIR, "checkpoint_*.pth")))
ema_files  = sorted(glob.glob(os.path.join(RUN_DIR, "ema_*.pth")))
all_pth    = sorted(glob.glob(os.path.join(RUN_DIR, "*.pth")))

total_bytes = 0

if not all_pth:
    print("  (no .pth files found)")
else:
    for fpath in all_pth:
        size_mb = os.path.getsize(fpath) / 1e6
        total_bytes += os.path.getsize(fpath)
        try:
            meta  = torch.load(fpath, weights_only=True)
            epoch = meta.get("epoch", "n/a")
        except Exception as exc:
            epoch = f"(load error: {exc})"
        fname = os.path.basename(fpath)
        print(f"  {fname:<30s}  epoch={epoch}  {size_mb:.1f} MB")

print("-" * 72)
print(f"  Total: {len(all_pth)} files  |  {total_bytes / 1e9:.2f} GB")
print()

# Quick summary
if ckpt_files:
    latest_meta = torch.load(ckpt_files[-1], weights_only=True)
    latest_ep   = latest_meta.get("epoch", "?")
    print(f"Latest checkpoint: epoch {latest_ep} ({os.path.basename(ckpt_files[-1])})")
    print(f"Epochs complete: {int(latest_ep)+1} / {EPOCHS}")
    remaining = EPOCHS - int(latest_ep) - 1
    print(f"Epochs remaining: {remaining}")
else:
    print("No checkpoint_*.pth files saved yet.")

## Cell 7 — Evaluation
Run `eval_obj.py` on the final EMA model to obtain FG-ARI and MBO on the CLEVRTex test set.

**Target (Table 1, AKOrN attn L=1):** FG-ARI = 75.6, MBO = 55.0

In [ ]:
import sys
import os
import subprocess

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

# ── Target checkpoint (EMA at final epoch 499) ────────────────────────────────
EVAL_MODEL_PATH = os.path.join(RUN_DIR, "ema_499.pth")

assert os.path.isfile(EVAL_MODEL_PATH), (
    f"Evaluation checkpoint not found: {EVAL_MODEL_PATH}\n"
    "Training may not be complete, or previous sessions' EMA files need to be restored. "
    "Attach the final session's output dataset and re-run Cell 2 to copy checkpoints."
)

print(f"Evaluating: {EVAL_MODEL_PATH}")
print("Target: FG-ARI = 75.6, MBO = 55.0  (Table 1, AKOrN attn L=1 on CLEVRTex)")
print("-" * 72)

# ── Main eval: CLEVRTex full test set (FG-ARI + MBO) ─────────────────────────
eval_cmd = [
    "python", os.path.join(REPO_DIR, "eval_obj.py"),
    f"--data_root={DATA_ROOT}",   # CLEVRTEX class appends clevrtex_full/ automatically
    "--model=akorn",
    "--data=clevrtex_full",
    "--J=attn",
    "--L=1",
    f"--model_path={EVAL_MODEL_PATH}",
    "--model_imsize=128",
]

print("Eval command:")
print(" ".join(eval_cmd))
print()

proc = subprocess.Popen(
    eval_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=REPO_DIR,
)

for line in proc.stdout:
    print(line, end="", flush=True)

proc.wait()

if proc.returncode != 0:
    raise subprocess.CalledProcessError(proc.returncode, eval_cmd)

print("-" * 72)
print("Evaluation complete.")

# ── Optional OOD and CAMO evaluations (commented out) ─────────────────────────
# Uncomment to evaluate on out-of-distribution splits:
#
# EVAL_OOD_CMD = [
#     "python", os.path.join(REPO_DIR, "eval_obj.py"),
#     f"--data_root={DATA_ROOT}",
#     "--model=akorn", "--data=clevrtex_outd",
#     "--J=attn", "--L=1",
#     f"--model_path={EVAL_MODEL_PATH}",
#     "--model_imsize=128",
# ]
#
# EVAL_CAMO_CMD = [
#     "python", os.path.join(REPO_DIR, "eval_obj.py"),
#     f"--data_root={DATA_ROOT}",
#     "--model=akorn", "--data=clevrtex_camo",
#     "--J=attn", "--L=1",
#     f"--model_path={EVAL_MODEL_PATH}",
#     "--model_imsize=128",
# ]
#
# for name, cmd in [("OOD", EVAL_OOD_CMD), ("CAMO", EVAL_CAMO_CMD)]:
#     print(f"\n--- {name} eval ---")
#     subprocess.run(cmd, check=True, cwd=REPO_DIR)